# Practice with LEFT JOINs – Finding Missing Data

INNER JOINs only return matching rows.  
**LEFT JOINs** keep *every* row from the left table and fill the right-side columns with `NULL` when there is no match.

This is the main tool for answering questions like:
- Which artists have **no** albums?
- Which customers have **never** bought anything?
- Which tracks are **not** in any playlist?
- Which employees support **zero** customers?

Use these exercises to get comfortable with that pattern.

## Data Schema (same as before)

![Database Schema](data-schema.png)

**Key relationships for this notebook:**

| Left Table     | Right Table      | Join Condition                          | Typical “missing” question              |
|----------------|------------------|-----------------------------------------|-----------------------------------------|
| `artists`      | `albums`         | `artists.ArtistId = albums.ArtistId`    | Artists with no albums                  |
| `customers`    | `invoices`       | `customers.CustomerId = invoices.CustomerId` | Customers who never purchased       |
| `tracks`       | `playlist_track` | `tracks.TrackId = playlist_track.TrackId` | Tracks never added to a playlist     |
| `employees`    | `customers`      | `employees.EmployeeId = customers.SupportRepId` | Employees who support no one     |
| `genres`       | `tracks`         | `genres.GenreId = tracks.GenreId`       | Genres with no tracks                   |
| `albums`       | `tracks`         | `albums.AlbumId = tracks.AlbumId`       | Albums with no tracks                   |

---
## Core Idea

```sql
SELECT left.*, right.*
FROM left_table AS left
LEFT JOIN right_table AS right
  ON left.key = right.key;
```

- Every row from `left_table` appears at least once.
- When there is no match, the columns from `right_table` are `NULL`.
- To find the *missing* rows, filter with `WHERE right.key IS NULL`.

---
## **Exercise 1: Artists and their albums (including artists with none)**

List every artist together with the titles of their albums.  
Artists who have never released an album should still appear (album title will be `NULL`).

### Instructions

Write a SQLite query that returns:

- Artist `Name`
- Album `Title` (will be NULL for artists with no albums)

Order by artist name, then album title.

**Hints:**

- Start from `artists` (the side you want to keep completely).
- `LEFT JOIN albums ON artists.ArtistId = albums.ArtistId`

---
## **Exercise 2: Find artists who have NO albums**

Now filter the previous result so that only artists without any albums remain.

### Instructions

Return only the artist `Name` for artists that have zero albums.

**Hints:**

- Same LEFT JOIN as Exercise 1.
- Add `WHERE albums.AlbumId IS NULL` (or any column from the right table).

---
## **Exercise 3: Customers and their purchase history**

Show every customer and the invoices they have (if any).

### Instructions

Return:

- Customer `FirstName`, `LastName`
- `InvoiceId` and `InvoiceDate` (NULL if the customer never bought anything)

Order by last name, first name, then invoice date.

**Hints:**

- Left table = `customers`
- `LEFT JOIN invoices ON customers.CustomerId = invoices.CustomerId`

---
## **Exercise 4: Customers who have NEVER purchased anything**

Marketing wants a list of customers who have not generated any revenue yet.

### Instructions

Return `CustomerId`, `FirstName`, `LastName`, `Email`  
only for customers who have no rows in the `invoices` table.

**Hints:**

- LEFT JOIN + `WHERE invoices.InvoiceId IS NULL`

---
## **Exercise 5: Tracks that are not in any playlist**

Some tracks may exist in the catalog but were never added to a playlist.

### Instructions

Return the `TrackId` and `Name` of tracks that do not appear in the `playlist_track` table.

**Hints:**

- Left table = `tracks`
- `LEFT JOIN playlist_track ON tracks.TrackId = playlist_track.TrackId`
- Filter where `playlist_track.TrackId IS NULL`

---
## **Exercise 6: Employees and the customers they support**

Show every employee together with the customers assigned to them as support representative.

### Instructions

Return:

- Employee full name (`FirstName || ' ' || LastName` as `Employee`)
- Customer full name (or NULL if the employee supports no one)

Order by employee name.

**Hints:**

- Join condition is `employees.EmployeeId = customers.SupportRepId`
- Start from `employees` so you keep employees who have zero customers.

---
## **Exercise 7: Employees who support NO customers**

Identify employees that currently have an empty support queue.

### Instructions

Return `EmployeeId`, full name, and `Title`  
only for employees who are not listed as `SupportRepId` for any customer.

**Hints:**

- Same LEFT JOIN as Exercise 6 + `WHERE customers.CustomerId IS NULL`

---
## **Exercise 8: Count albums per artist (including zero)**

How many albums does each artist have?  
Artists with zero albums must still appear with a count of 0.

### Instructions

Return:

- Artist `Name`
- `COUNT(albums.AlbumId)` as `AlbumCount`

Group by artist and order by `AlbumCount` ascending (so the zeros appear first).

**Important note:**

```sql
COUNT(albums.AlbumId)   -- counts only non-NULL values → gives 0 for missing matches
COUNT(*)                -- would count the left-table row itself → gives 1 even when there is no album
```

Always use `COUNT(right_table.primary_key)` when you want a true zero for missing matches.

---
## **Exercise 9: Genres with the fewest (or zero) tracks**

Find genres that have very few or no tracks associated with them.

### Instructions

Return genre `Name` and the number of tracks (`TrackCount`).  
Include genres that have zero tracks.  
Order by `TrackCount` ascending and show the 10 smallest.

**Hints:**

- `genres LEFT JOIN tracks ON genres.GenreId = tracks.GenreId`
- `COUNT(tracks.TrackId) AS TrackCount`
- `GROUP BY genres.Name`

---
## **Exercise 10: Albums that contain no tracks**

(This should return an empty result in the standard Chinook data, but the pattern is still useful to practice.)

### Instructions

Return album `Title` and artist name for any albums that have zero tracks.

**Hints:**

- Start from `albums`, LEFT JOIN `tracks`, then also join `artists` if you want the artist name.
- Filter `WHERE tracks.TrackId IS NULL`

---
## Bonus Challenge

### Customers who have never bought a track from a specific genre (e.g. “Rock”)

Write a query that finds customers who have **never** purchased any track belonging to the genre “Rock”.

*Approach:*
1. First find all customers who *have* bought at least one Rock track (INNER JOIN path through invoices → invoice_items → tracks → genres).
2. Then use a LEFT JOIN (or NOT IN / NOT EXISTS) against the full customers table to find those who are missing from that set.

Try both the LEFT JOIN + IS NULL pattern and a `NOT EXISTS` version if you feel comfortable.

---
## Quick Reference – LEFT JOIN Patterns

```sql
-- 1. Keep all rows from the left table
SELECT a.*, b.*
FROM left_table a
LEFT JOIN right_table b ON a.key = b.key;

-- 2. Find rows in left that have NO match in right
SELECT a.*
FROM left_table a
LEFT JOIN right_table b ON a.key = b.key
WHERE b.key IS NULL;

-- 3. Count including zeros
SELECT a.Name, COUNT(b.Id) AS cnt          -- use the right-side PK
FROM left_table a
LEFT JOIN right_table b ON a.key = b.key
GROUP BY a.Name
ORDER BY cnt;

-- 4. Multi-table LEFT JOIN (still keep all left rows)
SELECT a.*, b.*, c.*
FROM a
LEFT JOIN b ON a.id = b.a_id
LEFT JOIN c ON b.id = c.b_id;
```

---
## When to use which join

| Goal                                      | Join type   |
|-------------------------------------------|-------------|
| Only matching rows                        | INNER JOIN  |
| All rows from left + matches from right   | LEFT JOIN   |
| All rows from right + matches from left   | RIGHT JOIN  |
| All rows from both (rare in practice)     | FULL OUTER  |
| Find missing / unmatched records          | LEFT JOIN + `IS NULL` |

In most real-world analytics work you will use **INNER JOIN** and **LEFT JOIN** far more than the others.

---
**Practice tip:**  
After writing a LEFT JOIN, always check a few rows where the right-side columns are NULL.  
Those are the “missing data” cases you are looking for.